# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
%pip -q install transformers sentencepiece torch


Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
task_to_family = {
    "Sentiment classification":   "Encoder-only (e.g. BERT, RoBERTa)",
    "Machine translation":        "Encoder-decoder (e.g. T5, BART, MarianMT)",
    "Open-ended text generation": "Decoder-only (e.g. GPT family)",
    "Named Entity Recognition":   "Encoder-only (e.g. BERT, RoBERTa)",
    "Text summarization":         "Encoder-decoder (e.g. T5, BART)",
}

for i, (task, family) in enumerate(task_to_family.items(), 1):
    print(f"{i}. {task:28} -> {family}")

1. Sentiment classification     -> Encoder-only (e.g. BERT, RoBERTa)
2. Machine translation          -> Encoder-decoder (e.g. T5, BART, MarianMT)
3. Open-ended text generation   -> Decoder-only (e.g. GPT family)
4. Named Entity Recognition     -> Encoder-only (e.g. BERT, RoBERTa)
5. Text summarization           -> Encoder-decoder (e.g. T5, BART)


**Reasoning**
- **Sentiment classification / NER** only need to *understand* the input (one label per sentence or per token) → **encoder-only**.
- **Translation / summarization** turn one input sequence into a *different* output sequence → **encoder–decoder**.
- **Open-ended generation** continues a prompt token by token → **decoder-only**.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"{sentence:45} -> label: {result['label']:8}  score: {result['score']:.4f}")

The movie was excellent.                      -> label: POSITIVE  score: 0.9999
The lecture was difficult to understand.      -> label: NEGATIVE  score: 0.9997
I enjoyed learning about transformers.        -> label: POSITIVE  score: 0.9997


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "Natural language processing (NLP) is a branch of artificial intelligence that helps computers "
    "understand, interpret and generate human language. It combines computational linguistics with "
    "machine learning and deep learning models. NLP is used in many everyday applications such as "
    "machine translation, chatbots, spam filters, voice assistants and sentiment analysis of reviews."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
summary_ids = model.generate(**inputs, max_new_tokens=40, num_beams=4)

print("Original paragraph:\n", paragraph)
print("\nSummary:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))

Original paragraph:
 Natural language processing (NLP) is a branch of artificial intelligence that helps computers understand, interpret and generate human language. It combines computational linguistics with machine learning and deep learning models. NLP is used in many everyday applications such as machine translation, chatbots, spam filters, voice assistants and sentiment analysis of reviews.

Summary:
 Use natural language processing (NLP) to help computers understand, interpret and generate human language.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A chatbot has to *generate* a reply, and decoder-only models such as GPT are trained exactly for that:
predicting the next token given all the previous tokens. The whole conversation (system prompt, previous
turns and the new question) can simply be placed in the context, and the model continues it token by
token, so a single architecture handles any free-form, open-ended response. Its causal (left-to-right)
attention matches the way text is produced, and it scales very well when trained on huge amounts of
text, which gives it broad knowledge and fluent language. Encoder-only models (BERT) are built to
understand/classify text, not to write long answers, so they are not suitable for a chatbot.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


### Optional Challenge – Solution

In [9]:
prompt = "Natural language processing is"

# Sampling ON, no fixed seed -> each run picks different tokens
for run in range(1, 4):
    out = generator(prompt, max_new_tokens=25, do_sample=True, top_k=50, temperature=0.9)
    print(f"Run {run}: {out[0]['generated_text']}\n")

Run 1: Natural language processing is not limited to some aspects:






















Run 2: Natural language processing is a tool that allows developers the ability to automate tasks, including manual testing. However, it is not a feature that comes with



Run 3: Natural language processing is a major step in the success of a language.





In a nutshell, the process of processing a



In [10]:
# With a fixed seed the "random" choices are reproducible
for run in range(1, 3):
    set_seed(42)
    out = generator(prompt, max_new_tokens=25, do_sample=True, top_k=50, temperature=0.9)
    print(f"Seeded run {run}: {out[0]['generated_text']}\n")

Seeded run 1: Natural language processing is an area that has gained significant popularity since the 1970s. Language processing in this area is currently under way. Several studies have



Seeded run 2: Natural language processing is an area that has gained significant popularity since the 1970s. Language processing in this area is currently under way. Several studies have



**Why do we get different outputs for the same prompt?**

With `do_sample=True` the model does not always choose the single most likely next token. It computes a
probability distribution over the vocabulary and **randomly samples** a token from it (here restricted to
the top-50 tokens and reshaped by the temperature). Because one different token changes the context for
every following step, the generated texts quickly diverge. With greedy decoding (`do_sample=False`) or
with a fixed random seed (`set_seed(42)`), the choice becomes deterministic and the output repeats.